In [ ]:
#nibabel installed for medical image handling
!pip install -q nibabel scipy

import os, glob, random, shutil
import numpy as np
import nibabel as nib
from scipy import ndimage

In [ ]:
# Mount Google Drive
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
INPUT_DIR = "/content/drive/MyDrive/cropped"

In [ ]:
# Patient ID lists (from clinical.csv)
mild_ids = [2,6,8,12,18,27]         # 12 patients
moderate_ids = [0,7,17,23,48]    # 11 patients
TARGET_PER_CLASS = 30

Below block, To check content inside input file for repetition

In [ ]:
import os   #to check content inside input file for repetition
INPUT_DIR = "/content/drive/MyDrive/cropped"

if not os.path.exists(INPUT_DIR):
    raise FileNotFoundError(f"Folder not found: {INPUT_DIR}")
file_list = os.listdir(INPUT_DIR)   # List all files in the folder
file_list.sort()  # Sort the files for easier viewing

print("Files in folder:") # Print all entries
for f in file_list:
    print(f)

Files in folder:
__MACOSX
cropped
pat0_cropped.nii.gz
pat0_cropped_seg.nii.gz
pat0_cropped_seg_endpoints.nii.gz
pat10_cropped.nii.gz
pat10_cropped_seg.nii.gz
pat10_cropped_seg_endpoints.nii.gz
pat11_cropped.nii.gz
pat11_cropped_seg.nii.gz
pat11_cropped_seg_endpoints.nii.gz
pat12_cropped.nii.gz
pat12_cropped_seg.nii.gz
pat12_cropped_seg_endpoints.nii.gz
pat13_cropped.nii.gz
pat13_cropped_seg.nii.gz
pat13_cropped_seg_endpoints.nii.gz
pat14_cropped.nii.gz
pat14_cropped_seg.nii.gz
pat14_cropped_seg_endpoints.nii.gz
pat15_cropped.nii.gz
pat15_cropped_seg.nii.gz
pat15_cropped_seg_endpoints.nii.gz
pat16_cropped.nii.gz
pat16_cropped_seg.nii.gz
pat16_cropped_seg_endpoints.nii.gz
pat17_cropped.nii.gz
pat17_cropped_seg.nii.gz
pat17_cropped_seg_endpoints.nii.gz
pat18_cropped.nii.gz
pat18_cropped_seg.nii.gz
pat18_cropped_seg_endpoints.nii.gz
pat19_cropped.nii.gz
pat19_cropped_seg.nii.gz
pat19_cropped_seg_endpoints.nii.gz
pat1_cropped.nii.gz
pat1_cropped_seg.nii.gz
pat1_cropped_seg_endpoints.nii.gz


Duplicates removed in the below block

In [ ]:
import os  #duplications removed

INPUT_DIR = "/content/drive/MyDrive/cropped"
file_list = os.listdir(INPUT_DIR)

# Filter only original .nii/.nii.gz files, ignore duplicates with (1)
original_files = [f for f in file_list if (f.endswith(".nii") or f.endswith(".nii.gz")) and "(1)" not in f]
original_files.sort()

print(f"Total original .nii/.nii.gz files: {len(original_files)}\n")
print("Original files:")
for f in original_files:
    print(f)

Total original .nii/.nii.gz files: 180

Original files:
pat0_cropped.nii.gz
pat0_cropped_seg.nii.gz
pat0_cropped_seg_endpoints.nii.gz
pat10_cropped.nii.gz
pat10_cropped_seg.nii.gz
pat10_cropped_seg_endpoints.nii.gz
pat11_cropped.nii.gz
pat11_cropped_seg.nii.gz
pat11_cropped_seg_endpoints.nii.gz
pat12_cropped.nii.gz
pat12_cropped_seg.nii.gz
pat12_cropped_seg_endpoints.nii.gz
pat13_cropped.nii.gz
pat13_cropped_seg.nii.gz
pat13_cropped_seg_endpoints.nii.gz
pat14_cropped.nii.gz
pat14_cropped_seg.nii.gz
pat14_cropped_seg_endpoints.nii.gz
pat15_cropped.nii.gz
pat15_cropped_seg.nii.gz
pat15_cropped_seg_endpoints.nii.gz
pat16_cropped.nii.gz
pat16_cropped_seg.nii.gz
pat16_cropped_seg_endpoints.nii.gz
pat17_cropped.nii.gz
pat17_cropped_seg.nii.gz
pat17_cropped_seg_endpoints.nii.gz
pat18_cropped.nii.gz
pat18_cropped_seg.nii.gz
pat18_cropped_seg_endpoints.nii.gz
pat19_cropped.nii.gz
pat19_cropped_seg.nii.gz
pat19_cropped_seg_endpoints.nii.gz
pat1_cropped.nii.gz
pat1_cropped_seg.nii.gz
pat1_cropped

In [ ]:
# Output folder created
OUT_DIR = "/content/drive/MyDrive/augmented_output"
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Helper functions
def load_nifti(path):
    img = nib.load(path)
    return img.get_fdata(dtype=np.float32), img.affine, img.header, img.get_data_dtype()

def save_nifti(arr, affine, header, path, dtype, interp=True):
    try:
        arr = arr.astype(dtype)
    except:
        arr = arr.astype(np.float32)
    nib.save(nib.Nifti1Image(arr, affine, header), path)

def augment(img, seg, endp):
    # Random params
    angles = [random.uniform(-15,15) for _ in range(3)]
    flips = [random.random()<0.5 for _ in range(3)]
    shifts = [random.uniform(-5,5) for _ in range(3)]
    scale = random.uniform(0.9,1.1)
    noise_std = np.std(img)*0.01

    # Geometric (same for all three)
    def transform(vol, order):
        out = vol.copy()
        for ax, f in enumerate(flips):
            if f: out = np.flip(out, ax)
        for angle, axes in zip(angles, [(1,2),(0,2),(0,1)]):
            out = ndimage.rotate(out, angle, axes=axes, reshape=False, order=order, mode="nearest")
        out = ndimage.shift(out, shifts, order=order, mode="nearest")
        return out

    img_t = transform(img, 1)
    seg_t = transform(seg, 0)
    endp_t = transform(endp, 0)

    # Intensity + noise for image only
    img_t = img_t*scale + np.random.normal(0, noise_std, img_t.shape).astype(np.float32)
    return img_t, seg_t, endp_t

def distribute_aug(ids, target):
    n = len(ids)
    need = target-n
    base, rem = divmod(need,n)
    counts = {pid: base+(1 if i<rem else 0) for i,pid in enumerate(ids)}
    return counts

In [ ]:
# Install tqdm for progress bar
!pip install -q tqdm
from tqdm.notebook import tqdm

# Compute augmentations per patient

mild_counts = distribute_aug(mild_ids, TARGET_PER_CLASS)
moderate_counts = distribute_aug(moderate_ids, TARGET_PER_CLASS)

# Function to augment a group of patients

def augment_group(ids_counts):
    for pid, n_aug in ids_counts.items():
        if n_aug == 0:
            continue

        # Paths to original files
        img_path = os.path.join(INPUT_DIR, f"pat{pid}_cropped.nii.gz")
        seg_path = os.path.join(INPUT_DIR, f"pat{pid}_cropped_seg.nii.gz")
        endp_path = os.path.join(INPUT_DIR, f"pat{pid}_cropped_seg_endpoints.nii.gz")

        # Load originals
        img, affine, header, dtype = load_nifti(img_path)
        seg, _, _, seg_dtype = load_nifti(seg_path)
        endp, _, _, endp_dtype = load_nifti(endp_path)

        # Augmentation loop with progress bar
        for i in tqdm(range(1, n_aug+1), desc=f"Patient {pid}", leave=False):
            img_aug, seg_aug, endp_aug = augment(img, seg, endp)

            # Save augmented files
            save_nifti(img_aug, affine, header, os.path.join(OUT_DIR, f"pat{pid}_aug{i}_cropped.nii.gz"), dtype)
            save_nifti(seg_aug, affine, header, os.path.join(OUT_DIR, f"pat{pid}_aug{i}_cropped_seg.nii.gz"), seg_dtype)
            save_nifti(endp_aug, affine, header, os.path.join(OUT_DIR, f"pat{pid}_aug{i}_cropped_seg_endpoints.nii.gz"), endp_dtype)

# Run augmentation for mild and moderate groups

for group_name, counts in [("mild", mild_counts), ("moderate", moderate_counts)]:
    print(f"Processing {group_name} patients...")
    augment_group(counts)

# Zip and download
import shutil
from google.colab import files

shutil.make_archive("/content/augmented_dataset", "zip", OUT_DIR)
files.download("/content/augmented_dataset.zip")

print(" Augmentation complete! Download triggered.")

Processing mild patients...


Patient 2:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import os

# Output folder
OUT_DIR = "/content/drive/MyDrive/augmented_output"

# List all augmented files
aug_files = [f for f in os.listdir(OUT_DIR) if f.endswith(".nii") or f.endswith(".nii.gz")]

print(f"Total augmented files: {len(aug_files)}\n")

# Count per class
def count_class_files(ids, files_list):
    count = 0
    for pid in ids:
        count += sum(1 for f in files_list if f.startswith(f"pat{pid}_"))
    return count

mild_count = count_class_files(mild_ids, aug_files)
moderate_count = count_class_files(moderate_ids, aug_files)

print(f"Total files for mild patients: {mild_count}")
print(f"Total files for moderate patients: {moderate_count}")

# print total per patient
print("\nFiles per patient:")
for pid in mild_ids + moderate_ids:
    num = sum(1 for f in aug_files if f.startswith(f"pat{pid}_"))
    print(f"pat{pid}: {num} files")
